# Day 45 — Practice Questions

## Principal Component Analysis

**Boston University · Data Science Program**

---

**12 questions in four parts.**

| Part | | Questions |
|---|---|---|
| **A** | By hand | 1 – 3 |
| **B** | Scaling and components | 4 – 6 |
| **C** | Reading what came out | 7 – 9 |
| **D** | Judgement | 10 – 12 |

Run the setup cell first. Every question is self-contained after that.

> Replace every `...` with your own code. Where a question asks you to *explain*, write
> two or three sentences in the markdown cell — the explanation is the point.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.simplefilter("ignore")

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import accuracy_score, adjusted_rand_score
from sklearn.datasets import load_wine, load_breast_cancer, load_digits, load_iris

TEAL, CORAL, VIOLET, BLUE, AMBER = "#1F9E86", "#E05C45", "#8268B5", "#3E7CB1", "#F0A530"
MUTED, GREY = "#7B8AA1", "#C9D2E0"
np.set_printoptions(precision=4, suppress=True)

# ---- Part A works on six points you can check with a ruler -------------
SIX = np.array([[2.0, 1.0], [3.0, 3.0], [4.0, 3.0],
                [5.0, 6.0], [6.0, 6.0], [7.0, 8.0]])

# ---- the rest uses real datasets ---------------------------------------
CX, CY = load_breast_cancer(return_X_y=True)
CN = load_breast_cancer().feature_names
print("six points:", SIX.shape, "   breast cancer:", CX.shape)

---
# Part A · By hand

## Question 1

Six points, small enough to check on paper:

```
(2,1)  (3,3)  (4,3)  (5,6)  (6,6)  (7,8)
```

**Centre them** (subtract the mean of each column), then build the covariance matrix
**yourself** with the formula

$$C_{jk} = \frac{1}{n-1}\sum_i (x_{ij} - \bar{x}_j)(x_{ik} - \bar{x}_k)$$

Print your matrix and check it against `np.cov`.

In [ ]:
mean = SIX.mean(axis=0)
X = SIX - mean
print("means:", mean)
print("centred:\n", X)

n = len(SIX)
C = ...                      # build it with the formula, not np.cov
print("\nmy C:\n", np.round(C, 4))
print("np.cov:\n", np.round(np.cov(SIX, rowvar=False), 4))
print("match:", ...)

## Question 2

Sweep the direction yourself. For angles 0° to 180°, project the centred points onto
`w = (cos θ, sin θ)` and measure the variance of the projection.

Print the best angle and its variance, plot the curve, and confirm that the variance at
your best angle equals `w.T @ C @ w`.

In [ ]:
angles = np.arange(0, 180, 0.25)
spreads = np.array([... for a in angles])

best = angles[spreads.argmax()]
print(f"best angle {best:.2f}°   variance {spreads.max():.4f}")

wb = np.array([np.cos(np.radians(best)), np.sin(np.radians(best))])
print("w.T C w  =", round(float(wb @ C @ wb), 4))

plt.plot(angles, spreads, color=BLUE, lw=2.5)
plt.xticks([0, 45, 90, 135, 180]); plt.xlabel("direction"); plt.ylabel("variance")
plt.show()

## Question 3

Now get the same answer from `np.linalg.eigh(C)` and from `sklearn`'s `PCA`.

Print all three directions and all three variances. Then explain, in your own words,
**what an eigenvector of the covariance matrix is**, using the curve you just plotted.

In [ ]:
evals, evecs = np.linalg.eigh(C)
order = evals.argsort()[::-1]
evals, evecs = evals[order], evecs[:, order]

ang = np.degrees(np.arctan2(evecs[1, 0], evecs[0, 0])) % 180
print(f"eigh   : {ang:.2f}°   variance {evals[0]:.4f}")
print(f"sweep  : {best:.2f}°   variance {spreads.max():.4f}")

p = PCA().fit(SIX)
ang_sk = ...
print(f"sklearn: {ang_sk:.2f}°   variance {p.explained_variance_[0]:.4f}")

print("\nPC1 . PC2 =", round(float(evecs[:, 0] @ evecs[:, 1]), 12))
print("eigenvalues sum to", round(float(evals.sum()), 4),
      " vs trace(C) =", round(float(np.trace(C)), 4))

*Your answer:*

---
# Part B · Scaling and components

## Question 4

Break PCA with a unit change.

Take the breast cancer data and run PCA **without scaling**. Print PC1's explained
variance ratio and the three columns it loads on most heavily. Then do it **with**
`StandardScaler`, and compare.

In [ ]:
for tag, D in [("WITHOUT scaling", CX), ("WITH scaling", ...)]:
    p = PCA().fit(D)
    print(tag)
    print(f"   PC1 holds {p.explained_variance_ratio_[0]:.4%}")
    for i in np.abs(p.components_[0]).argsort()[::-1][:3]:
        print(f"     {CN[i]:26} loading {p.components_[0][i]:+.4f}")

## Question 5

How many components does the breast cancer data deserve?

Draw the scree plot, print how many components reach 80%, 90% and 95%, and then check
those thresholds against what a model actually needs.

In [ ]:
p = PCA().fit(CZ)
cum = np.cumsum(p.explained_variance_ratio_)

for thr in [0.80, 0.90, 0.95]:
    print(f"{thr:.0%} of the variance: {np.argmax(cum >= thr) + 1} components")

plt.bar(range(1, 31), p.explained_variance_ratio_ * 100, color=BLUE)
plt.plot(range(1, 31), cum * 100, color=CORAL, lw=2.2, marker="o", ms=3)
plt.xlabel("component"); plt.ylabel("% of variance"); plt.show()

print(f"\n{'k':>4}{'variance':>11}{'accuracy':>11}")
for k in [1, 2, 5, 10, 17, 30]:
    a = cross_val_score(LogisticRegression(max_iter=5000),
                        ..., CY, cv=5).mean()
    print(f"{k:>4}{cum[k-1]:>11.1%}{a:>11.4f}")

## Question 6

You have now seen a 95% threshold ask for 40 components on the digits (where 20 were
enough) and 10 here (where keeping all 30 still scored highest).

Explain **why a variance threshold cannot tell you how many components to keep**, and
describe what you would do instead. Be specific about what you would measure.

*Your answer:*

---
# Part C · Reading what came out

## Question 7

Name the components.

Run PCA on the scaled breast cancer data and print PC1's loadings sorted by size. Then
look at the column names and say, in one sentence, what PC1 appears to be measuring.

In [ ]:
p = PCA().fit(CZ)
order = np.abs(p.components_[0]).argsort()[::-1]
for i in order[:10]:
    print(f"  {CN[i]:28} {p.components_[0][i]:+.4f}")

print("\nsmallest loadings:")
for i in order[-4:]:
    print(f"  {CN[i]:28} {p.components_[0][i]:+.4f}")

print("\nall loadings the same sign?", ...)

*What does PC1 appear to measure?*

## Question 8

Rebuild the digits from a handful of numbers.

Load the digits, scale them, and reconstruct six images from 1, 3, 8 and 25 components.
Print the reconstruction error and the variance kept at each step. Show the images.

In [ ]:
DX, DY = load_digits(return_X_y=True)
scaler = StandardScaler().fit(DX)
DZ = scaler.transform(DX)
picks = [np.where(DY == d)[0][0] for d in [0, 2, 4, 6, 7, 9]]

fig, axes = plt.subplots(5, 6, figsize=(7, 6.4))
for j, idx in enumerate(picks):
    axes[0][j].imshow(DX[idx].reshape(8, 8), cmap="gray_r", vmin=0, vmax=16)

for i, k in enumerate([1, 3, 8, 25], start=1):
    pk = PCA(k).fit(DZ)
    back = ...                        # reconstruct, then undo the scaling
    err = np.sqrt(((DX - back) ** 2).mean())
    print(f"{k:3d} components  keeps {pk.explained_variance_ratio_.sum():6.1%}"
          f"  error {err:.3f}")
    for j, idx in enumerate(picks):
        axes[i][j].imshow(back[idx].reshape(8, 8), cmap="gray_r", vmin=0, vmax=16)

for row in axes:
    for ax in row:
        ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

## Question 9

Use PCA as a lens rather than a model.

The digits have 64 columns, so you cannot plot them. Reduce to 2 components, plot all
1,797 of them coloured by their true digit, and then say what the picture tells you that
a table of accuracy numbers would not.

In [ ]:
S2 = PCA(2).fit_transform(DZ)

plt.figure(figsize=(6.4, 5.4))
for d in range(10):
    m = DY == d
    plt.scatter(S2[m, 0], S2[m, 1], s=9, label=str(d))
plt.legend(ncol=5, fontsize=8, frameon=False)
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.show()

print("variance kept:", ...)

*Your answer:*

---
# Part D · Judgement

## Question 10

Build a dataset where PCA throws away the answer.

Make three columns: two with large spread that say nothing about the label, and one with
small spread that determines it completely. Show that keeping only PC1 destroys the
model, and that keeping all three components does not.

In [ ]:
rg = np.random.default_rng(99)
n = 500
label = rg.integers(0, 2, n)
noise1 = ...
noise2 = ...
signal = ...
X3 = np.c_[noise1, noise2, signal]

p = PCA().fit(X3)
print("column sds:", X3.std(axis=0, ddof=1).round(3))
print("explained variance ratio:", p.explained_variance_ratio_.round(4))

S3 = p.transform(X3)
for k in [1, 2, 3]:
    a = cross_val_score(LogisticRegression(max_iter=5000), S3[:, :k], label, cv=5).mean()
    print(f"keeping {k} component(s): accuracy {a:.4f}")

## Question 11

PCA before clustering: does it help?

Take the scaled digits and run `KMeans(10)` on the full 64 columns, and on 2, 10 and 20
components. Compare agreement with the true digits, using `adjusted_rand_score`.

In [ ]:
print(f"{'input':>18}{'agreement':>12}")
km = KMeans(10, n_init=10, random_state=0).fit(DZ)
print(f"{'all 64 columns':>18}{adjusted_rand_score(DY, km.labels_):>12.4f}")

for k in [2, 10, 20]:
    Sk = PCA(k, random_state=0).fit_transform(DZ)
    lab = ...
    print(f"{str(k) + ' components':>18}{adjusted_rand_score(DY, lab):>12.4f}")

*Your answer:*

## Question 12

**Argue against PCA.**

A colleague proposes running PCA on every dataset the team works with, as a standard first
step, "because it removes noise and speeds everything up".

Write **four or five sentences** against that policy, using evidence from this notebook.
Then say the one situation in which you would agree with them.

*Your argument:*


*When you would agree:*

---

## Submission

Run **Cell → Run All** on a fresh kernel and confirm there are no errors, then upload the
notebook. Questions 3, 6, 9 and 12 are marked mostly on the explanation — a correct number
with no reasoning gets little credit.

Solutions will be released with Day 46.